<a href="https://colab.research.google.com/github/varunsmn87/SolarcycleDTW/blob/main/DTW_Paper_Revision_2_Ver_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# DTW ANALOG SOLAR-CYCLE FORECAST
# Data freeze: CUT_OFF=2024-12-01 | RNG=42
# Framework: Walk-Forward Validation, Walk-Forward Residuals, Dual Conformal Calibration
# ============================================================

!pip -q install fastdtw

import os, warnings, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from fastdtw import fastdtw
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr
import matplotlib.patches as mpatches

warnings.filterwarnings("ignore")

# -----------------------------
# 0) Global config
# -----------------------------
CUT_OFF = pd.Timestamp("2024-12-01")
RNG_SEED = 42
np.random.seed(RNG_SEED)

OUT_FIG = "results/figures"
OUT_TAB = "results/tables"
os.makedirs(OUT_FIG, exist_ok=True)
os.makedirs(OUT_TAB, exist_ok=True)

# Cleaner typography
plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

STAMP = f"Data ≤ {CUT_OFF.date()} | RNG={RNG_SEED}"

def true_rmse(y, yhat):
    return float(np.sqrt(mean_squared_error(y, yhat)))

def safe_save(fig, stem):
    """Save both PDF and EPS without layout overlap."""
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_FIG, f"{stem}.pdf"), bbox_inches="tight")
    fig.savefig(os.path.join(OUT_FIG, f"{stem}.eps"), bbox_inches="tight")
    plt.close(fig)

# -----------------------------
# 1) Data loading + cycle cuts
# -----------------------------
FROZEN_DIR = "data_frozen"
os.makedirs(FROZEN_DIR, exist_ok=True)
FROZEN_SILSO_CSV = os.path.join(FROZEN_DIR, f"SILSO_SN_ms_tot_V2_until_{CUT_OFF.date()}.csv")

SILSO_URL = "https://www.sidc.be/SILSO/DATA/SN_ms_tot_V2.0.csv"

if os.path.exists(FROZEN_SILSO_CSV):
    df = pd.read_csv(FROZEN_SILSO_CSV, parse_dates=["Date"]).set_index("Date")
else:
    raw = pd.read_csv(SILSO_URL, header=None, delimiter=";")
    raw.columns = ["Year","Month","DateFraction","SmoothedSSN","Definitive","Extra1","Extra2"]
    raw["Date"] = pd.to_datetime(raw["Year"].astype(str) + "-" + raw["Month"].astype(str), errors="coerce")
    df = raw.dropna(subset=["Date"]).set_index("Date")
    df = df[df["SmoothedSSN"] >= 0].loc[:CUT_OFF].copy()
    df.reset_index().to_csv(FROZEN_SILSO_CSV, index=False)

cycle_boundaries = {
    13: ("1889-03-01", "1901-02-01"), 14: ("1901-03-01", "1913-07-01"),
    15: ("1913-08-01", "1923-07-01"), 16: ("1923-08-01", "1933-02-01"),
    17: ("1933-03-01", "1944-01-01"), 18: ("1944-02-01", "1954-03-01"),
    19: ("1954-04-01", "1964-09-01"), 20: ("1964-10-01", "1976-02-01"),
    21: ("1976-03-01", "1986-08-01"), 22: ("1986-09-01", "1996-07-01"),
    23: ("1996-08-01", "2008-12-01"), 24: ("2009-01-01", "2019-12-01"),
}

cycle_fits = {}
for sc, (st, en) in cycle_boundaries.items():
    seg = df.loc[pd.to_datetime(st):pd.to_datetime(en), "SmoothedSSN"].values.astype(float)
    cycle_fits[sc] = (np.arange(len(seg)), seg)

def _norm01(y):
    y = np.asarray(y, float)
    lo, hi = float(np.min(y)), float(np.max(y))
    if hi - lo < 1e-12: return np.zeros_like(y)
    return (y - lo) / (hi - lo)

def convex_ref(n):
    return np.sin(np.pi * np.arange(n) / max(1, n-1))

# -----------------------------
# 2) Core DTW pipeline
# -----------------------------
def dtw_select(target_curve, pool_cycles, k):
    t = _norm01(target_curve)
    dist_list = []
    for c in pool_cycles:
        yc = _norm01(cycle_fits[c][1])
        yc_adj = np.interp(np.linspace(0,1,len(t)), np.linspace(0,1,len(yc)), yc)
        d, _ = fastdtw(t, yc_adj)
        dist_list.append((c, float(d)))
    dist_list.sort(key=lambda z: z[1])
    chosen = dist_list[:min(k, len(dist_list))]
    sel = [c for c,_ in chosen]
    w = np.array([1.0/(d+1e-9) for _,d in chosen], float)
    w /= w.sum()
    return sel, w, dist_list

def blend_shape_norm(selected_cycles, w, target_len):
    M = []
    for c in selected_cycles:
        yc = _norm01(cycle_fits[c][1])
        yc_full = np.interp(np.linspace(0,1,target_len), np.linspace(0,1,len(yc)), yc)
        M.append(yc_full)
    M = np.vstack(M)
    return w @ M

def apply_idw_scaling(shape01, selected_cycles, w):
    amps = np.array([float(np.max(cycle_fits[c][1]) - np.min(cycle_fits[c][1])) for c in selected_cycles])
    bases = np.array([float(np.min(cycle_fits[c][1])) for c in selected_cycles])
    s = np.clip(shape01, 0, 1)
    return s * np.sum(w*amps) + np.sum(w*bases)

def apply_climatology_scaling(shape01, pool_cycles):
    peaks = [float(np.max(cycle_fits[c][1])) for c in pool_cycles]
    bases = [float(np.min(cycle_fits[c][1])) for c in pool_cycles]
    base_clim = float(np.median(bases))
    peak_clim = float(np.median(peaks))
    s = np.clip(shape01, 0, 1)
    return base_clim + s*(peak_clim-base_clim)

# -----------------------------
# 3) Phase-aware residual bootstrap
# -----------------------------
BINS = 12

def phase_bin_indices(n, bins=BINS):
    idx = (np.linspace(0,1,n,endpoint=True)*bins).astype(int)
    return np.minimum(idx, bins-1)

def build_phase_residual_bins_walkforward(pool_cycles, k_val, prefix_len=60):
    bins_raw = [[] for _ in range(BINS)]
    for cyc in pool_cycles:
        train = [c for c in range(13, cyc) if c % 2 == cyc % 2]
        if len(train) < 1: continue

        _, y = cycle_fits[cyc]
        m = min(prefix_len, len(y))

        sel, w, _ = dtw_select(y[:m], train, min(k_val, len(train)))
        sh = blend_shape_norm(sel, w, len(y))
        yhat = apply_idw_scaling(sh, sel, w)
        yhat += (y[m-1] - yhat[m-1])

        n = min(len(y), len(yhat))
        r = y[:n] - yhat[:n]
        idx = phase_bin_indices(n, bins=BINS)
        for t in range(n):
            bins_raw[idx[t]].append(float(r[t]))

    bins_zero = []
    for b in bins_raw:
        if len(b)==0: bins_zero.append([])
        else:
            mu = float(np.mean(b))
            bins_zero.append([x - mu for x in b])
    return bins_zero

def bootstrap_pi(forecast, bins_zero, draws=5000, q=(0.10, 0.90), seed=0):
    rng = np.random.default_rng(seed)
    f = np.asarray(forecast, float)
    N = len(f)
    idx = phase_bin_indices(N, bins=len(bins_zero))
    lo = np.empty(N, float)
    hi = np.empty(N, float)

    # 1. Clip negatives to prevent NaN in sqrt
    f_pos = np.clip(f, 0.0, None)
    f_max = np.max(f_pos) if np.max(f_pos) > 0 else 1.0

    # 2. Poisson-inspired scaling with a basal variance floor (35%)
    ramp = np.sqrt(f_pos / f_max)
    ramp = np.clip(ramp, 0.35, 1.0)

    for t in range(N):
        pool = np.asarray(bins_zero[idx[t]], float)
        if pool.size == 0:
            lo[t], hi[t] = f[t], f[t]
        else:
            samp = (rng.choice(pool, size=draws, replace=True) * ramp[t]) + f[t]
            lo[t] = max(0.0, float(np.quantile(samp, q[0])))
            hi[t] = float(np.quantile(samp, q[1]))
    return lo, hi



# -----------------------------
# 4) Validation + BEST_K_RT (Real-Time Walk-Forward SC21-24)
# -----------------------------
PREFIX_LEN = 60
k_values = [1, 2, 3, 4]

def eval_cycle_rt_emulation(sc, k):
    _, y = cycle_fits[sc]
    pool = [c for c in range(13, sc) if c % 2 == sc % 2]
    m = min(PREFIX_LEN, len(y))
    sel, w, _ = dtw_select(y[:m], pool, k)
    sh = blend_shape_norm(sel, w, len(y))
    yhat = apply_idw_scaling(sh, sel, w)
    yhat += (y[m-1] - yhat[m-1])
    return float(mean_absolute_error(y, yhat)), true_rmse(y, yhat), float(r2_score(y, yhat)), float(pearsonr(y, yhat)[0])

val_rows = []
for k in k_values:
    metrics = [eval_cycle_rt_emulation(sc, k) for sc in range(21,25)]
    val_rows.append((k, float(np.mean([m[0] for m in metrics])), float(np.mean([m[1] for m in metrics])), float(np.mean([m[2] for m in metrics])), float(np.mean([m[3] for m in metrics]))))

val_df = pd.DataFrame(val_rows, columns=["k","MAE","RMSE","R2","r"]).sort_values("MAE")
BEST_K_RT = int(val_df.iloc[0]["k"])

print("Real-Time Validation Summary by k (SC21-24):")
print(val_df.to_string(index=False))
print(f"\nSelected BEST_K_RT = {BEST_K_RT}\n")

# -----------------------------
# 4b) A Priori Validation + BEST_K_AP (Baseline Comparison SC21-24)
# -----------------------------
def forecast_offline_convex(pool_cycles, k, target_len, scaling="climatology"):
    ref = convex_ref(target_len)
    sel, w, dist_list = dtw_select(ref, pool_cycles, k)
    sh = blend_shape_norm(sel, w, target_len)
    if scaling == "climatology":
        f = apply_climatology_scaling(sh, pool_cycles)
    else:
        f = apply_idw_scaling(sh, sel, w)
    return f, sel, w, dist_list

def eval_cycle_apriori_emulation(sc, k):
    _, y = cycle_fits[sc]
    pool = [c for c in range(13, sc) if c % 2 == sc % 2]
    mean_len = int(round(np.mean([len(cycle_fits[c][1]) for c in pool])))
    yhat, _, _, _ = forecast_offline_convex(pool, k, mean_len, scaling="climatology")
    n = min(len(y), len(yhat))
    return float(mean_absolute_error(y[:n], yhat[:n])), true_rmse(y[:n], yhat[:n]), float(r2_score(y[:n], yhat[:n])), float(pearsonr(y[:n], yhat[:n])[0])

apriori_val_rows = []
for k in k_values:
    metrics = [eval_cycle_apriori_emulation(sc, k) for sc in range(21, 25)]
    apriori_val_rows.append((k, float(np.mean([m[0] for m in metrics])), float(np.mean([m[1] for m in metrics])), float(np.mean([m[2] for m in metrics])), float(np.mean([m[3] for m in metrics]))))

apriori_val_df = pd.DataFrame(apriori_val_rows, columns=["k","MAE","RMSE","R2","r"]).sort_values("MAE")
BEST_K_AP = int(apriori_val_df.iloc[0]["k"])

print("A Priori Validation Summary by k (SC21-24):")
print(apriori_val_df.to_string(index=False))
print(f"\nSelected BEST_K_AP = {BEST_K_AP}\n")

# -----------------------------
# 5) Dual-Quantile Conformal Calibration
# -----------------------------
odd_pool = [c for c in cycle_fits if c%2==1 and c<=23]
even_pool = [c for c in cycle_fits if c%2==0 and c<=24]

# Calibrate using the RT hyperparameter since SC25 is the primary target
odd_bins0 = build_phase_residual_bins_walkforward(odd_pool, BEST_K_RT, PREFIX_LEN)

def _val_cov(scale, target_q, cycles=(21,23)):
    bins_s = [[scale * x for x in b] for b in odd_bins0]
    covs = []
    for sc in cycles:
        _, y = cycle_fits[sc]
        pool = [c for c in range(13, sc) if c % 2 == sc % 2]
        if len(pool) == 0: continue
        m = min(PREFIX_LEN, len(y))
        sel, w, _ = dtw_select(y[:m], pool, min(BEST_K_RT, len(pool)))
        sh = blend_shape_norm(sel, w, len(y))
        yhat = apply_idw_scaling(sh, sel, w)
        yhat += (y[m-1] - yhat[m-1])
        lo, hi = bootstrap_pi(yhat, bins_s, q=target_q, seed=RNG_SEED+sc)
        covs.append(float(np.mean((y >= lo) & (y <= hi))))
    return float(np.mean(covs))

grid = np.linspace(0.5, 3.0, 51)
s80 = float(min(grid, key=lambda s: abs(_val_cov(s, target_q=(0.10, 0.90)) - 0.80)))
odd_bins_80 = [[s80 * x for x in b] for b in odd_bins0]
s98 = float(min(grid, key=lambda s: abs(_val_cov(s, target_q=(0.01, 0.99)) - 0.98)))
odd_bins_98 = [[s98 * x for x in b] for b in odd_bins0]

print(f"[CAL] Conformal scale s80={s80:.2f} | val 80% cov={_val_cov(s80, target_q=(0.10,0.90)):.3f}")
print(f"[CAL] Conformal scale s98={s98:.2f} | val 98% cov={_val_cov(s98, target_q=(0.01,0.99)):.3f}\n")

# -----------------------------
# 6) Forecast helpers
# -----------------------------
def forecast_realtime(sc_obs, pool_cycles, k):
    y_obs = np.asarray(sc_obs, float)
    t_len = len(y_obs)
    sel, w, dist_list = dtw_select(y_obs, pool_cycles, k)
    target_len = max(t_len, int(round(np.mean([len(cycle_fits[c][1]) for c in sel]))))
    sh = blend_shape_norm(sel, w, target_len)
    f = apply_idw_scaling(sh, sel, w)
    f += (y_obs[-1] - f[t_len-1])
    return f, sel, w, dist_list, target_len

def cosine_blend(a, b, blend_len=6):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    L = int(blend_len)
    if L <= 0: return np.concatenate([a, b])
    L = min(L, len(a), len(b))
    w = 0.5*(1 - np.cos(np.linspace(0, np.pi, L)))
    mid = (1-w)*a[-L:] + w*b[:L]
    return np.concatenate([a[:-L], mid, b[L:]])

# -----------------------------
# 7) Build Forecasts
# -----------------------------
sc25_start = (pd.to_datetime(cycle_boundaries[24][1]) + pd.offsets.MonthBegin(1))
y_obs_25 = df.loc[sc25_start:CUT_OFF, "SmoothedSSN"].values.astype(float)
t_len = len(y_obs_25)

# SC25 Real-Time (Uses BEST_K_RT)
f25_rt, sel25_rt, w25_rt, dist25_rt, L25 = forecast_realtime(y_obs_25, odd_pool, BEST_K_RT)
f25_rt = np.clip(f25_rt, 0.0, None)
lo80_25, hi80_25 = bootstrap_pi(f25_rt, odd_bins_80, q=(0.10,0.90), seed=RNG_SEED+80)
lo98_25, hi98_25 = bootstrap_pi(f25_rt, odd_bins_98, q=(0.01,0.99), seed=RNG_SEED+98)

cov80 = float(np.mean((y_obs_25 >= lo80_25[:t_len]) & (y_obs_25 <= hi80_25[:t_len])))
cov98 = float(np.mean((y_obs_25 >= lo98_25[:t_len]) & (y_obs_25 <= hi98_25[:t_len])))

print(f"SC25 Real-Time Forecast (60 months obs | k={BEST_K_RT}):")
print(f"  Analogs: {sel25_rt} | Peak: {np.max(f25_rt):.2f} at month {int(np.argmax(f25_rt))}")
print(f"  Empirical Cov (80% PI): {100*cov80:.2f}% | (98% PI): {100*cov98:.2f}%\n")

# SC25 A Priori (Uses BEST_K_AP)
mean_len_odd = int(round(np.mean([len(cycle_fits[c][1]) for c in odd_pool])))
f25_off, sel25_off, w25_off, dist25_off = forecast_offline_convex(odd_pool, BEST_K_AP, mean_len_odd, scaling="climatology")
f25_off = np.clip(f25_off, 0.0, None)
lo98_25off, hi98_25off = bootstrap_pi(f25_off, odd_bins_98, q=(0.01,0.99), seed=RNG_SEED+250)

print(f"SC25 A Priori Forecast (k={BEST_K_AP}):")
print(f"  Analogs: {sel25_off} | Peak: {np.max(f25_off):.2f} at month {int(np.argmax(f25_off))}\n")

# SC26 A Priori (Uses BEST_K_AP)
even_bins0 = build_phase_residual_bins_walkforward(even_pool, BEST_K_AP, PREFIX_LEN)
even_bins_98 = [[s98 * x for x in b] for b in even_bins0]
mean_len_even = int(round(np.mean([len(cycle_fits[c][1]) for c in even_pool])))
f26_off, sel26_off, w26_off, dist26_off = forecast_offline_convex(even_pool, BEST_K_AP, mean_len_even, scaling="climatology")
f26_off = np.clip(f26_off, 0.0, None)
lo98_26off, hi98_26off = bootstrap_pi(f26_off, even_bins_98, q=(0.01,0.99), seed=RNG_SEED+260)

print(f"SC26 A Priori Forecast (k={BEST_K_AP}):")
print(f"  Analogs: {sel26_off} | Peak: {np.max(f26_off):.2f} at month {int(np.argmax(f26_off))}\n")

# Stitch SC25+26
f25_26 = cosine_blend(f25_rt, f26_off, blend_len=6)

# -----------------------------
# 8) FIGURES
# -----------------------------
def fig_full_series_with_cycles():
    fig, ax = plt.subplots(figsize=(11.5, 3.6))
    ax.plot(df.index, df["SmoothedSSN"].values, linewidth=1.0)
    for sc, (st, en) in cycle_boundaries.items():
        st = pd.to_datetime(st); en = pd.to_datetime(en)
        ax.axvline(st, linewidth=0.6)
        mid = st + (en - st)/2
        ax.text(mid, ax.get_ylim()[1]*0.93, f"SC{sc}", ha="center", va="top", fontsize=9, rotation=0)

    ax.axvline(sc25_start, linestyle="--", linewidth=1.0)
    y_top = ax.get_ylim()[1]
    ax.annotate("SC25 start", xy=(sc25_start, y_top*0.88), xytext=(sc25_start + pd.DateOffset(years=3), y_top*1.03), textcoords="data", arrowprops=dict(arrowstyle="->", linewidth=1.0), ha="left", va="bottom", fontsize=9, clip_on=False)
    ax.set_ylim(ax.get_ylim()[0], y_top*1.08)
    ax.set_title("Monthly Smoothed Sunspot Number (SILSO) with Solar Cycle Labels")
    ax.set_ylabel("Smoothed SSN")
    ax.set_xlabel("Year")
    ax.text(0.01, 0.02, STAMP, transform=ax.transAxes, fontsize=9)
    ax.margins(x=0.01)
    return fig
safe_save(fig_full_series_with_cycles(), "fig01_full_series_cycle_labels")

def fig_normalized_cycles_grid(sc_start=13, sc_end=24):
    cycles = list(range(sc_start, sc_end + 1))
    fig, axes = plt.subplots(3, 4, figsize=(11.5, 7), sharex=True, sharey=True)
    axes = axes.ravel()
    for i, sc in enumerate(cycles):
        ax = axes[i]
        _, y = cycle_fits[sc]
        ax.plot(np.linspace(0, 1, len(y)), _norm01(y), linewidth=1.6)
        ax.set_title(f"SC{sc}", fontsize=10)
        ax.set_xlim(0,1); ax.set_ylim(0,1)
        ax.tick_params(labelsize=8)
    plt.subplots_adjust(left=0.08, right=0.98, bottom=0.12, top=0.92, hspace=0.35, wspace=0.25)
    fig.suptitle("Time- and Amplitude-Normalized Solar Cycles (SC13–SC24)", fontsize=12)
    fig.supxlabel("Normalized Time (τ)", fontsize=11, y=0.04)
    fig.supylabel("Normalized SSN (0–1)", fontsize=11, x=0.02)
    fig.text(0.01,0.01,STAMP,fontsize=9)
    return fig
safe_save(fig_normalized_cycles_grid(13,24), "fig02_normalized_cycles_grid_sc13_24")

def fig_validation_by_k(v_df, title):
    fig, ax = plt.subplots(figsize=(6.2, 3.4))
    x = np.arange(len(v_df))
    ax.bar(x, v_df["MAE"].values)
    ax.set_xticks(x); ax.set_xticklabels([f"k={int(k)}" for k in v_df["k"].values])
    ax.set_title(title)
    ax.set_ylabel("Avg MAE")
    for i,v in enumerate(v_df["MAE"].values): ax.text(i, v, f"{v:.2f}", ha="center", va="bottom", fontsize=9)
    ax.text(0.01, 0.02, STAMP, transform=ax.transAxes, fontsize=9)
    return fig
safe_save(fig_validation_by_k(val_df.sort_values("k"), "Walk-Forward Real-Time Emulation (SC21–SC24): Avg MAE by k"), "fig03a_realtime_validation_mae_by_k")
safe_save(fig_validation_by_k(apriori_val_df.sort_values("k"), "Walk-Forward A Priori Emulation (SC21–SC24): Avg MAE by k"), "fig03b_apriori_validation_mae_by_k")

def fig_residual_histograms(bins_zero, title):
    fig, axes = plt.subplots(3, 4, figsize=(11.2, 6.2), constrained_layout=True)
    axes = axes.ravel()
    for i in range(12):
        data = np.asarray(bins_zero[i], float)
        if data.size > 0: axes[i].hist(data, bins=30)
        axes[i].set_title(f"Bin {i+1}", fontsize=10)
        axes[i].axvline(0.0, linewidth=1.0)
    fig.suptitle(title, fontsize=12)
    fig.text(0.01, 0.01, STAMP, fontsize=9)
    return fig
safe_save(fig_residual_histograms(odd_bins_98, "Zero-Mean Walk-Forward Residual Histograms (Odd Cycles)"), "fig04a_odd_residual_hists_12bins")
safe_save(fig_residual_histograms(even_bins_98, "Zero-Mean Walk-Forward Residual Histograms (Even Cycles)"), "fig04b_even_residual_hists_12bins")

def fig_dtw_distances(dist_list, pool_cycles, title):
    rows = [(c,d) for (c,d) in dist_list if c in pool_cycles]
    cycles = [c for c,_ in rows]
    dists  = [d for _,d in rows]
    fig, ax = plt.subplots(figsize=(6.4, 3.4))
    ax.bar([str(c) for c in cycles], dists)
    ax.set_title(title)
    ax.set_xlabel("Candidate cycle")
    ax.set_ylabel("DTW distance")
    ax.tick_params(axis="x", rotation=0)
    ax.text(0.01, 0.02, STAMP, transform=ax.transAxes, fontsize=9)
    return fig
safe_save(fig_dtw_distances(dist25_rt, odd_pool, "DTW Distances: Real-Time Observed SC25 Prefix vs. Odd Cycles"), "fig05a_dtw_distances_sc25_rt")
safe_save(fig_dtw_distances(dist26_off, even_pool, "DTW Distances: A Priori Reference Shape vs. Even Cycles"), "fig05b_dtw_distances_sc26_apriori")

def fig_forecast_with_pi(f, lo, hi, title, obs=None, obs_len=None, band_label="Prediction interval", band_face="0.92", band_edge="0.65", band_hatch="////"):
    fig, ax = plt.subplots(figsize=(7.2, 3.6))
    x = np.arange(len(f))
    ax.fill_between(x, lo, hi, facecolor=band_face, edgecolor=band_edge, hatch=band_hatch, linewidth=0.6, zorder=1)
    f_line, = ax.plot(x, f, linewidth=2.0, label="Forecast", zorder=3)
    obs_line = None
    if obs is not None:
        ox = np.arange(obs_len if obs_len is not None else len(obs))
        obs_line, = ax.plot(ox, obs, linewidth=1.6, color="black", label="Observed", zorder=4)
    ax.set_title(title); ax.set_xlabel("Months since cycle start"); ax.set_ylabel("Smoothed SSN")
    ax.text(0.01, 0.02, STAMP, transform=ax.transAxes, fontsize=9)
    b_proxy = mpatches.Patch(facecolor=band_face, edgecolor=band_edge, hatch=band_hatch, label=band_label)
    handles = [f_line, b_proxy]
    if obs_line is not None: handles.insert(1, obs_line)
    ax.legend(handles=handles, frameon=False, loc="upper right")
    return fig

safe_save(fig_forecast_with_pi(f25_off, lo98_25off, hi98_25off, f"A priori forecast of Solar Cycle 25 (k={BEST_K_AP}) with 98% phase-aware PI."), "fig06_sc25_offline_98pi")
safe_save(fig_forecast_with_pi(f25_rt, lo98_25, hi98_25, f"Real-time forecast of Solar Cycle 25 (k={BEST_K_RT}, Jan 2020–Dec 2024 obs) with 98% PI", obs=y_obs_25, obs_len=t_len), "fig07_sc25_realtime_98pi")
safe_save(fig_forecast_with_pi(f26_off, lo98_26off, hi98_26off, f"Forecasted Solar Cycle 26 (A priori, k={BEST_K_AP}) with 98% PI"), "fig09_sc26_offline_98pi")

def fig_stitched(f_stitch, cut_at=None, title="Combined Forecast for Solar Cycles 25 and 26 (Smooth Transition)"):
    fig, ax = plt.subplots(figsize=(8.0, 3.6))
    ax.plot(np.arange(len(f_stitch)), f_stitch, linewidth=1.8)
    if cut_at is not None: ax.axvline(cut_at, linestyle="--", linewidth=1.0)
    ax.set_title(title); ax.set_xlabel("Months since SC25 start"); ax.set_ylabel("Smoothed SSN")
    ax.text(0.01, 0.02, STAMP, transform=ax.transAxes, fontsize=9)
    return fig
safe_save(fig_stitched(f25_26, cut_at=len(f25_rt)), "fig10_sc25_sc26_stitched")

# -----------------------------
# 9) Tables: SC25 summary
# -----------------------------
sc25_summary = pd.DataFrame([{
    "BEST_K_RT": BEST_K_RT,
    "BEST_K_AP": BEST_K_AP,
    "s80": s80,
    "s98": s98,
    "SC25_RT_Analogs": str(sel25_rt),
    "SC25_RT_Peak": float(np.max(f25_rt)),
    "SC25_RT_PeakMonth": int(np.argmax(f25_rt)),
    "Cov80_obs": cov80,
    "Cov98_obs": cov98,
    "RMSE_obs_window": true_rmse(y_obs_25, f25_rt[:t_len]),
    "MAE_obs_window": float(mean_absolute_error(y_obs_25, f25_rt[:t_len])),
    "R2_obs_window": float(r2_score(y_obs_25, f25_rt[:t_len])),
    "r_obs_window": float(pearsonr(y_obs_25, f25_rt[:t_len])[0]),
}])
sc25_summary.to_csv(os.path.join(OUT_TAB, "table_sc25_realtime_summary.csv"), index=False)

print("All figures saved to:", OUT_FIG)
print("All tables saved to:", OUT_TAB)
print("\nDONE.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
Real-Time Validation Summary by k (SC21-24):
 k       MAE      RMSE       R2        r
 3 21.466390 26.891261 0.789847 0.958176
 4 23.747688 28.864911 0.769294 0.957685
 2 26.917992 32.893124 0.720369 0.938347
 1 30.169673 36.552459 0.639430 0.900816

Selected BEST_K_RT = 3

A Priori Validation Summary by k (SC21-24):
 k       MAE      RMSE       R2        r
 2 26.393647 31.809688 0.646637 0.962927
 1 27.225356 33.433588 0.624130 0.948444
 4 27.225391 32.845934 0.617241 0.955275
 3 27.542962 32.617452 0.625141 0.956729

Selected BEST_K_AP = 2

[CAL] Conformal scale s80=1.50 | val 80% cov=0.795
[CAL] Conformal scale s98=2.60 | val 98% cov=0.977

SC25 Real-Time Forecast (60 months obs | k=3):
  Analogs: [17, 15, 21] | Peak: 173.35 at month 48
  Empirical Cov (80% PI): 91.67% | (98% PI): 100.00%

SC25 A Priori Forecast (k=2):
  Analogs: [19, 21] | Peak: 188.69 at month 

All figures saved to: results/figures
All tables saved to: results/tables

DONE.


In [2]:
# -----------------------------
# 10) Extra SC25 Figures
# -----------------------------

# Figure 11: Normalized Analogs vs Observation
def fig_sc25_normalized_analogs():
    fig, ax = plt.subplots(figsize=(7.2, 3.6))

    # Plot normalized SC25 obs
    y_norm = _norm01(y_obs_25)
    ax.plot(np.linspace(0, 1, len(y_norm)), y_norm,
            color="black", linewidth=2.5, label="SC25 Obs (Norm)", zorder=5)

    # Plot normalized analogs
    colors = plt.cm.viridis(np.linspace(0, 0.9, len(sel25_rt)))
    for i, sc in enumerate(sel25_rt):
        _, y = cycle_fits[sc]
        y_c_norm = _norm01(y)
        ax.plot(np.linspace(0, 1, len(y_c_norm)), y_c_norm,
                color=colors[i], linewidth=1.5, linestyle="--", label=f"SC{sc} (Norm)")

    ax.set_title("Morphological Comparison: SC25 Obs vs. Real-Time Analogs")
    ax.set_xlabel("Normalized Time (τ)")
    ax.set_ylabel("Normalized Amplitude")
    ax.legend(frameon=False, fontsize=9)
    ax.text(0.01, 0.02, STAMP, transform=ax.transAxes, fontsize=9)
    return fig

safe_save(fig_sc25_normalized_analogs(), "fig11_sc25_normalized_analogs")

# Figure 12: Forecast Comparison (Real-Time vs A Priori)
def fig_sc25_forecast_comparison():
    fig, ax = plt.subplots(figsize=(7.2, 3.6))

    # Plot Obs
    ax.plot(np.arange(len(y_obs_25)), y_obs_25, color="black", linewidth=2.0, label="Observed", zorder=5)

    # Plot Real-Time Forecast
    ax.plot(np.arange(len(f25_rt)), f25_rt, color="tab:blue", linewidth=2.0, label=f"Real-Time (k={BEST_K_RT})", zorder=4)

    # Plot A Priori Forecast
    # Align start? Both start at t=0 relative to cycle start.
    ax.plot(np.arange(len(f25_off)), f25_off, color="tab:red", linewidth=1.5, linestyle="--", label=f"A Priori (k={BEST_K_AP})", zorder=3)

    ax.set_title("SC25 Forecast Evolution: A Priori vs. Real-Time")
    ax.set_xlabel("Months since SC25 start")
    ax.set_ylabel("Smoothed SSN")
    ax.legend(frameon=False)
    ax.text(0.01, 0.02, STAMP, transform=ax.transAxes, fontsize=9)
    return fig

safe_save(fig_sc25_forecast_comparison(), "fig12_sc25_forecast_comparison")

print("Extra figures saved.")

Extra figures saved.


In [3]:
# ---------------------------------------------------------
# 10) Retrospective Operational Evolution of SC25 Real-Time Forecast
# ---------------------------------------------------------
import matplotlib.cm as cm

print("\n=== 10. Simulating Retrospective Operational Forecast Evolution ===")

def generate_forecast_for_cutoff(cutoff_str):
    """Generates the real-time forecast as if we were standing at the cutoff date."""
    dt_cutoff = pd.to_datetime(cutoff_str)

    # Use data only up to the simulated cutoff and drop NAs to find true max date
    sim_series = df.loc[sc25_start:dt_cutoff, "SmoothedSSN"].dropna()

    if len(sim_series) == 0:
        raise ValueError(f"No observed SC25 data available up to cutoff {cutoff_str}")

    y_obs_sim = sim_series.values.astype(float)
    actual_cutoff_date = sim_series.index.max()

    # Run the exact same real-time pipeline, explicitly using BEST_K_RT (k=3)
    f_rt_sim, sel_sim, w_sim, _, _ = forecast_realtime(y_obs_sim, odd_pool, BEST_K_RT)

    # Apply physical constraint: SSN cannot be negative
    f_rt_sim = np.clip(f_rt_sim, a_min=0.0, a_max=None)

    peak_val = float(np.max(f_rt_sim))
    peak_idx = int(np.argmax(f_rt_sim))

    # Calculate calendar-aware peak date
    peak_date = sc25_start + pd.DateOffset(months=peak_idx)

    return y_obs_sim, f_rt_sim, sel_sim, peak_val, peak_date, actual_cutoff_date

# Define historical operational cutoffs
cutoffs = ["2020-12-01", "2021-12-01", "2022-12-01", "2023-12-01", "2024-12-01"]
evolution_data = {}
table_rows = []

# 1. Add A Priori Baseline (Uses f25_off generated previously with BEST_K_AP, k=2)
f25_off_clamped = np.clip(f25_off, a_min=0.0, a_max=None)
apriori_peak_idx = int(np.argmax(f25_off_clamped))
apriori_peak_date = sc25_start + pd.DateOffset(months=apriori_peak_idx)

table_rows.append({
    "Target Cutoff": "A Priori",
    "Actual Data End": "Pre-SC25",
    "Months Obs.": 0,
    "Mode": f"Ref. Shape (k={BEST_K_AP})",
    "Analogs": ", ".join(map(str, sel25_off)),
    "Peak SSN": round(np.max(f25_off_clamped), 1),
    "Peak Date": apriori_peak_date.strftime('%Y-%m')
})

# 2. Loop through historical real-time cutoffs
for c_date in cutoffs:
    y_sim, f_sim, analogs_sim, p_val, p_date, actual_cutoff = generate_forecast_for_cutoff(c_date)
    evolution_data[c_date] = {
        "y_obs": y_sim,
        "f_rt": f_sim,
        "actual_cutoff": actual_cutoff
    }
    table_rows.append({
        "Target Cutoff": c_date,
        "Actual Data End": actual_cutoff.strftime('%Y-%m-%d'),
        "Months Obs.": len(y_sim),
        "Mode": f"Obs. Prefix (k={BEST_K_RT})",
        "Analogs": ", ".join(map(str, analogs_sim)),
        "Peak SSN": round(p_val, 1),
        "Peak Date": p_date.strftime('%Y-%m')
    })

    # PROOF OF ANCHOR EQUALITY
    anchor_gap = abs(f_sim[len(y_sim) - 1] - y_sim[-1])
    assert np.isclose(f_sim[len(y_sim) - 1], y_sim[-1], atol=1e-4), f"Anchor gap failed for {c_date}: {anchor_gap}"
    print(f"Forecast updated {actual_cutoff.strftime('%Y-%m')} (Obs: {len(y_sim)}m): Peak = {p_val:.1f} on {p_date.strftime('%Y-%m')} | Analogs: {analogs_sim}")

# 3. Export Summary Table
evol_df = pd.DataFrame(table_rows)
evol_df.to_csv(os.path.join(OUT_TAB, "table_sc25_forecast_evolution.csv"), index=False)
evol_df.to_latex(os.path.join(OUT_TAB, "table_sc25_forecast_evolution.tex"), index=False)
print("\nForecast Evolution Table:\n", evol_df.to_string(index=False))

# 4. Generate Figure 16
def fig_forecast_evolution(evol_dict, apriori_f, full_obs):
    fig, ax = plt.subplots(figsize=(10.0, 6.0))

    obs_end = df.loc[sc25_start:CUT_OFF, "SmoothedSSN"].dropna().index.max()
    truth_label = f"Observed SC25 (to {obs_end.strftime('%b %Y')})"

    ax.plot(np.arange(len(full_obs)), full_obs, color="black", linewidth=2.2, label=truth_label, zorder=5)
    ax.plot(np.arange(len(apriori_f)), apriori_f, color="gray", linestyle=":", linewidth=2.0, label=f"A Priori Baseline (k={BEST_K_AP})", zorder=2)

    colors = cm.plasma(np.linspace(0.1, 0.8, len(evol_dict)))

    for i, c_date in enumerate(sorted(evol_dict.keys())):
        data = evol_dict[c_date]
        f_rt = data["f_rt"]
        y_obs = data["y_obs"]
        actual_cutoff = data["actual_cutoff"]
        t_len = len(y_obs)

        if t_len == 0: continue

        label = f"Updated {actual_cutoff.strftime('%b %Y')} (k={BEST_K_RT})"

        # STRICT BRANCHING
        future_y = f_rt[t_len - 1:]
        future_x = np.arange(t_len - 1, t_len - 1 + len(future_y))

        ax.plot(future_x, future_y, color=colors[i], linestyle="--", linewidth=2.0, label=label, zorder=12+i)
        ax.scatter(t_len - 1, y_obs[-1], color=colors[i], s=55, zorder=20, edgecolor="white", linewidth=1.0)

    ax.set_title("Retrospective Operational Evolution of the SC25 Forecast")
    ax.set_xlabel("Months since SC25 start")
    ax.set_ylabel("Smoothed SSN")
    ax.legend(loc="upper left", fontsize=9)
    ax.grid(True, alpha=0.25)

    ax.set_ylim(bottom=0)
    max_len = max(len(apriori_f), len(full_obs), *[len(v["f_rt"]) for v in evol_dict.values()])
    ax.set_xlim(0, max_len - 1)

    ax.text(0.01, 0.02, STAMP, transform=ax.transAxes, fontsize=9)
    return fig

safe_save(fig_forecast_evolution(evolution_data, f25_off_clamped, y_obs_25), "fig16_sc25_forecast_evolution")
print("\nGenerated fig16_sc25_forecast_evolution.eps cleanly.")


=== 10. Simulating Retrospective Operational Forecast Evolution ===
Forecast updated 2020-12 (Obs: 12m): Peak = 208.8 on 2024-01 | Analogs: [17, 15, 19]
Forecast updated 2021-12 (Obs: 24m): Peak = 184.6 on 2024-01 | Analogs: [17, 15, 21]
Forecast updated 2022-12 (Obs: 36m): Peak = 177.5 on 2024-01 | Analogs: [17, 15, 21]
Forecast updated 2023-12 (Obs: 48m): Peak = 131.2 on 2024-01 | Analogs: [17, 15, 21]
Forecast updated 2024-12 (Obs: 60m): Peak = 173.4 on 2024-01 | Analogs: [17, 15, 21]

Forecast Evolution Table:
 Target Cutoff Actual Data End  Months Obs.              Mode    Analogs  Peak SSN Peak Date
     A Priori        Pre-SC25            0  Ref. Shape (k=2)     19, 21     188.7   2024-02
   2020-12-01      2020-12-01           12 Obs. Prefix (k=3) 17, 15, 19     208.8   2024-01
   2021-12-01      2021-12-01           24 Obs. Prefix (k=3) 17, 15, 21     184.6   2024-01
   2022-12-01      2022-12-01           36 Obs. Prefix (k=3) 17, 15, 21     177.5   2024-01
   2023-12-01     


Generated fig16_sc25_forecast_evolution.eps cleanly.


In [4]:
# ---------------------------------------------------------
# SENSITIVITY CHECK: SC25 Real-Time Evolution with
# ---------------------------------------------------------
print("\n=== SENSITIVITY SNEAK PEEK: SC25 Evolution with k=2 ===")
for c_date in cutoffs:
    dt_cutoff = pd.to_datetime(c_date)
    sim_series = df.loc[sc25_start:dt_cutoff, "SmoothedSSN"].dropna()

    if len(sim_series) == 0:
        continue

    y_obs_sim = sim_series.values.astype(float)

    # Run the real-time forecast strictly forcing k=2
    f_rt_k2, sel_k2, _, _, _ = forecast_realtime(y_obs_sim, odd_pool, k=2)
    f_rt_k2 = np.clip(f_rt_k2, a_min=0.0, a_max=None)

    p_val_k2 = float(np.max(f_rt_k2))
    p_date_k2 = sc25_start + pd.DateOffset(months=int(np.argmax(f_rt_k2)))

    print(f"Cutoff {c_date} (Obs: {len(y_obs_sim):02d}m) | Peak = {p_val_k2:.1f} on {p_date_k2.strftime('%Y-%m')} | Analogs: {sel_k2}")
print("=======================================================\n")


=== SENSITIVITY SNEAK PEEK: SC25 Evolution with k=2 ===
Cutoff 2020-12-01 (Obs: 12m) | Peak = 184.8 on 2024-02 | Analogs: [17, 15]
Cutoff 2021-12-01 (Obs: 24m) | Peak = 181.6 on 2024-02 | Analogs: [17, 15]
Cutoff 2022-12-01 (Obs: 36m) | Peak = 187.8 on 2024-02 | Analogs: [17, 15]
Cutoff 2023-12-01 (Obs: 48m) | Peak = 133.4 on 2024-02 | Analogs: [17, 15]
Cutoff 2024-12-01 (Obs: 60m) | Peak = 172.6 on 2024-02 | Analogs: [17, 15]

